In [1]:
import functools as f
from datetime import datetime

from sklearn.cluster import SpectralClustering
from numpy.typing import NDArray
import numpy as np

from common import grid_search, load_data, cross_validate

In [2]:
def get_clusters(
    adj_matrix: NDArray[np.float64 | np.int32], hyperparameters: dict, seed=456
):
    clustering = SpectralClustering(
        n_clusters=hyperparameters["n_clusters"],
        assign_labels=hyperparameters["strategy"],
        random_state=seed,
    ).fit(adj_matrix)

    return clustering.labels_

In [3]:
def generate_hyperparameters_for_sc(max_number_clusters: int):
    combinations = []

    for strategy in ["kmeans", "discretize", "cluster_qr"]:
        combinations.append(
            {
                "strategy": strategy,
            }
        )

    return combinations

In [4]:
method = "sc"
dataset = "dwug_es"
path_to_gold_data = "./gold-data-es.csv" 
path_to_data= f"./deepmistake_model_es.csv"
max_number_clusters = 5
model = "deepmistake"

In [5]:
metadata = {
    "method": method,
    "dataset": dataset,
    "path_to_data": path_to_data,
    "path_to_gold_data": path_to_gold_data,
    "fill_diagonal": True,
    "normalize": True,
    "model": model,
    "use_threshold": True,
}

In [6]:
start_time = datetime.now()

# grid_search(
#     f.partial(load_data, path_to_data),
#     get_clusters,
#     generate_hyperparameters_for_sc(max_number_clusters=max_number_clusters),
#     metadata=metadata,
# )

print(f"Elapsed time: {datetime.now() - start_time}")

Elapsed time: 0:00:00.000057


## Cross-validation experiments

In [7]:
gold_dir = "./dwug_es_cleaned/clusters"

In [8]:
start_time = datetime.now()

cv_summary = cross_validate(
    get_clusters,
    generate_hyperparameters_for_sc(max_number_clusters=max_number_clusters),
    metadata=metadata,
    gold_dir=gold_dir,
    k=5,
)

print(f"Elapsed time: {datetime.now() - start_time}")

2026-06-13 16:08:41,792 - INFO - loading data from ./deepmistake_model_es.csv ...
2026-06-13 16:08:42,114 - INFO - data loaded ...
2026-06-13 16:08:42,127 - INFO - processing fold_1 ...
2026-06-13 16:08:42,134 - INFO - get predictions ...
2026-06-13 16:08:42,140 - INFO - building adjacency matrix ...
2026-06-13 16:08:42,158 - INFO - adjacency matrix built ...
/home/fzamora/miniconda3/envs/ex/lib/python3.12/site-packages/sklearn/cluster/_spectral.py:703: UserWarning: The spectral clustering API has changed. ``fit``now constructs an affinity matrix from data. To use a custom affinity matrix, set ``affinity=precomputed``.
  warnings.warn(
2026-06-13 16:08:42,479 - INFO -  n_clusters=2 silhouette=0.0321
/home/fzamora/miniconda3/envs/ex/lib/python3.12/site-packages/sklearn/cluster/_spectral.py:703: UserWarning: The spectral clustering API has changed. ``fit``now constructs an affinity matrix from data. To use a custom affinity matrix, set ``affinity=precomputed``.
  warnings.warn(
2026-06-1

FileNotFoundError: [Errno 2] No such file or directory: 'dwug_es_cleaned/clusters/atrás.csv'

In [ ]:
print(f"\nProtocol 1 (ARI driven): ")
print(f"  avg test ARI: {cv_summary['protocol_ari']['avg_test_ari']:.4f}")
print(f"  avg test LSCD: {cv_summary['protocol_ari']['avg_test_lscd']:.4f}")
print(f"\nProtocol 2 (LSCD Driven):")
print(f"  avg test LSCD: {cv_summary['protocol_lscd']['avg_test_lscd']:.4f}")
print(f"  avg test ARI: {cv_summary['protocol_lscd']['avg_test_ari']:.4f}")